# 10-демо · LLM ичинде: кийинки сөздү божомолдоо

**Кыргыз Республикасынын Эсептөө палатасы · AI тренинги · 3-күн · тирүү демонстрация**

> 🎤 **Бул дептер — тренер үчүн.**

**Тренингге ЧЕЙИН бир жолу даярдык (интернет керек):** дептерди толук иштетип чыгыңыз —
GPT-2 модели (~350 МБ) жүктөлөт. Андан кийин интернетсиз иштейт.
Анан жаңы HTML резерв жасаңыз: `jupyter nbconvert --to html 10_llm.ipynb`

ChatGPT сыяктуу системалардын түпкү механизмин **өз көзүбүз менен** көрөбүз:
модель ар кадамда бир гана нерсе кылат — **кийинки сөздү божомолдойт**.

## 0-кадам · Даярдык

In [ ]:
%pip install -q transformers torch

In [ ]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

токенайзер = GPT2Tokenizer.from_pretrained("distilgpt2")
модель = GPT2LMHeadModel.from_pretrained("distilgpt2")
модель.eval()
print("GPT-2 (кичине версиясы) даяр ✓")
print("Параметрлери:", sum(p.numel() for p in модель.parameters()) // 1_000_000, "миллион")
print("(Салыштыруу: ChatGPT артындагы моделдер — жүздөгөн миллиард)")

## 1-кадам · Моделдин «оюн столун» ачабыз

Сүйлөм башын беребиз — модель кийинки сөзгө канча ыктымалдуулук берерин көрөбүз.
(Модель англисче окутулган — мисалдар англисче, экранда чечмелеп беребиз.)

In [ ]:
def топ5(суйлом_башы):
    киргизүү = токенайзер(суйлом_башы, return_tensors="pt")
    with torch.no_grad():
        логиттер = модель(**киргизүү).logits[0, -1]
    ыктымал = torch.softmax(логиттер, dim=-1)
    топ = torch.topk(ыктымал, 5)
    print(f"«{суйлом_башы} ...»")
    for б, и in zip(топ.values, топ.indices):
        сөз = токенайзер.decode(и).strip()
        тилке = "█" * int(б * 120)
        print(f"   {сөз:<12} {б:6.1%}  {тилке}")
    print()

топ5("The auditor checked the")
топ5("The capital of France is")
топ5("Two plus two equals")

Модель «билет» окшойт, ээ? Жок — ал миллиарддаган тексттен **кайсы сөз кайсынын артынан келерин** гана эсептеген.

## 2-кадам · Кадам сайын: текст кантип «өсөт»

Эң ыктымалдуу сөздү тандап, кайра баштан божомолдоп — текст ушинтип жаралат:

In [ ]:
текст = "The audit report shows"
киргизүү = токенайзер(текст, return_tensors="pt")

for кадам in range(15):
    with torch.no_grad():
        логиттер = модель(**киргизүү).logits[0, -1]
    кийинки = int(torch.argmax(логиттер))
    киргизүү["input_ids"] = torch.cat([киргизүү["input_ids"], torch.tensor([[кийинки]])], dim=1)
    киргизүү["attention_mask"] = torch.ones_like(киргизүү["input_ids"])

print(токенайзер.decode(киргизүү["input_ids"][0]))

## 3-кадам · «Температура»: тобокелчил жана этият модель

Ар дайым эң ыктымалдуусун тандаса — текст кайталанчаак болот. Ошондуктан системалар
бир аз «кокустук» кошот. Муну **температура** дейт:

In [ ]:
def жарат(баш, температура, узундук=18):
    ки = токенайзер(баш, return_tensors="pt")
    for _ in range(узундук):
        with torch.no_grad():
            л = модель(**ки).logits[0, -1] / температура
        кийинки = int(torch.multinomial(torch.softmax(л, dim=-1), 1))
        ки["input_ids"] = torch.cat([ки["input_ids"], torch.tensor([[кийинки]])], dim=1)
        ки["attention_mask"] = torch.ones_like(ки["input_ids"])
    return токенайзер.decode(ки["input_ids"][0])

torch.manual_seed(7)
print("Температура 0.3 (этият):")
print(" ", жарат("The government budget for education", 0.3))
print()
print("Температура 1.2 (тобокелчил):")
print(" ", жарат("The government budget for education", 1.2))

## 4-кадам · Галлюцинация кайдан чыгат — өз көзүбүз менен

Моделден **жок нерсени** сурайлы. Ал «билбейм» дебейт — ыктымалдуу угулган текстти жаза берет:

In [ ]:
torch.manual_seed(3)
print(жарат("The president of the Moon Republic is", 0.8, 20))
print()
print("Модель 'Ай Республикасы' жок экенин 'билбейт' — ал жөн гана ыктымалдуу")
print("текстти улантты. ЧОҢ моделдер да ушул эле принципте иштейт — алар да")
print("ишенимдүү ЖАҢЫЛА алат. Кечээги сабак: ишенимдүү ката — эң коркунучтуу ката.")

## 5-кадам · ⭐ ТИРҮҮ: чоң LLM менен чат

Эми браузерде чыныгы чоң модель менен сүйлөшөбүз (ChatGPT, Claude же башка).
Залга көрсөтө турган 3 эксперимент:

1. **Пайдалуу жагы:** «Мына бул абзацты расмий стилге которуп бер» — черновик секунддарда
2. **Галлюцинация тести:** атайын жок нерсени сурайбыз (мис. ойдон чыгарылган мыйзамдын беренеси) — канчалык ишенимдүү жооп берерин көрөбүз
3. **Купуялуулук эскертүүсү:** чатка эч кандай кызматтык маалымат жазбайбыз — экранда да!

⚠️ Бул бөлүк браузерде өтөт — дептерде код жок.

## Жыйынтык

✓ LLM = «кийинки сөздү божомолдо» деген жөнөкөй оюндун АБДАН чоң версиясы

✓ Температура — тактык менен чыгармачылыктын ортосундагы тандоо

⚠️ Галлюцинация — ката эмес, механизмдин табияты: модель чындыкты эмес, ыктымалдуу текстти жазат

⚠️ Фактыларды текшерүү — дайыма адамдын милдети

**Курс аяктады — жыйынтыктоого өтөбүз! 🎉**